In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

In [ ]:
#Uncomment and run if needed

#

# Data Cleansing & Anonymization
We will be using Spark to load data from a dataset about people in NSS
We are going to clean data from a file that contains data about different employees, with several columns, among them:
* first name
* middle name
* last name
* ssn (Social security number)
* birthdate
* salary
* company
* department
* role
There are several issues:
* Name format: first name may not be right, for instance we may have Rachel or CAROL
* SSN format: Some of the records come without hyphens. for instance  628-40-9043 and 324469083
* Due to the format issues, data come with duplicates, your mission is to fix it.
Once the issues are fixed, records are guaranteed to match, so you can remove duplicates easily
* Also you should anonymize SSN

In [ ]:
from pyspark.sql.functions import split, col

file_path = #<TODO>

raw_df = spark.read.csv(file_path,
                         header="true", 
                          inferSchema="true")

raw_df.count()

In [ ]:
from pyspark.sql.functions import concat, substring, lit

# A function that fix SSN to proper format XXX-XX-XXXX
def format_ssn(ssn):
    #<TODO>

# Filter Columns not containing a hyphen in the ssn (bad rows)
# Create a new column with the same name fixing the ssn column
wrong_ssn_df = raw_df.filter(~col(<TODO>).contains(<TODO>))\ 
                .withColumn(<TODO>, format_ssn(col(<TODO>))) 


# Filter again the original dataframe to keep only the rows containing a - (good rows)
# Union it with the dataset with the fixed SSN
fixed_ssn_df = raw_df.filter(col("ssn").contains("-")) \
.union(wrong_ssn_df)

In [ ]:
from pyspark.sql.functions import  initcap

#Use initcap function, that giving a String returns it with inly the first letter in uppercase
#No need to bifurcate dataset this time since good rows are not afected by the fix, they stay the same
fixed_name_df = fixed_ssn_df \
.withColumn(<TODO>)

In [ ]:
#Now that all the rows are clear, we can safely remove duplicates, use drop_duplicates method without arguments for that
fixed_df = fixed_name_df.<TODO>

fixed_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import sha2

#Now we want to anonymize the salary and the ssn
#Use sha2 function with the column and the number of bits(256) inside withColumn
#Beware that it only applies on String, you need to cast salary to string first (or at the time)
anon_df = fixed_df \
  .withColumn("salary", sha2(col("salary").cast("string"), 256)) \
  .withColumn("ssn", sha2(col("ssn"), 256))


anon_df.show(truncate=False)